In [1]:
import csv
from itertools import combinations
from collections import defaultdict, Counter

class FPTreeNode:
    """Узел FP-дерева"""
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None

    def increment(self, count):
        self.count += count


def build_fp_tree(transactions, min_support_count, item_counts):
    """Строит FP-дерево"""
    # Фильтруем частые товары
    frequent_items = {item: count for item, count in item_counts.items() 
                     if count >= min_support_count}
    
    if not frequent_items:
        return None, None
    
    # Сортируем по убыванию частоты
    sorted_items = sorted(frequent_items.items(), key=lambda x: (-x[1], x[0]))
    
    # Создаем корневой узел
    root = FPTreeNode(None, 0, None)
    header_table = {}
    
    # Инициализируем таблицу заголовков
    for item, count in sorted_items:
        header_table[item] = [count, None]
    
    # Строим дерево
    for transaction in transactions:
        # Фильтруем и сортируем товары в транзакции
        filtered_items = [item for item in transaction if item in frequent_items]
        if not filtered_items:
            continue
            
        # Сортируем по убыванию частоты
        filtered_items.sort(key=lambda x: (-frequent_items[x], x))
        
        # Вставляем в дерево
        current_node = root
        for item in filtered_items:
            if item in current_node.children:
                child = current_node.children[item]
                child.increment(1)
            else:
                child = FPTreeNode(item, 1, current_node)
                current_node.children[item] = child
                
                # Обновляем таблицу заголовков
                if header_table[item][1] is None:
                    header_table[item][1] = child
                else:
                    node = header_table[item][1]
                    while node.link is not None:
                        node = node.link
                    node.link = child
            
            current_node = child
    
    return root, header_table, frequent_items


def ascend_tree(node):
    """Поднимается от узла к корню, возвращая путь"""
    path = []
    while node.parent is not None:
        path.append(node.item)
        node = node.parent
    return path[::-1]  # Переворачиваем, чтобы получить путь от корня


def find_conditional_patterns(item, header_table):
    """Находит условные паттерны для товара"""
    patterns = []
    node = header_table[item][1]
    
    while node is not None:
        if node.count > 0:
            path = ascend_tree(node.parent)  # Исключаем сам item
            if path:
                patterns.append((path, node.count))
        node = node.link
    
    return patterns


def mine_fp_tree(header_table, min_support_count, prefix, 
                 frequent_itemsets, total_transactions, item_counts):
    """Рекурсивно добывает частые наборы"""
    # Сортируем товары по частоте
    items = [item for item in header_table.keys()]
    items.sort(key=lambda x: item_counts[x])
    
    for item in items:
        # Создаем новый набор
        new_freq_set = prefix + [item]
        new_freq_set_tuple = tuple(sorted(new_freq_set))
        
        # Поддержка набора равна поддержке последнего добавленного товара
        # (так как все товары в наборе частые)
        support = item_counts[item] / total_transactions
        
        # Сохраняем набор
        k = len(new_freq_set_tuple)
        if k not in frequent_itemsets:
            frequent_itemsets[k] = {}
        frequent_itemsets[k][new_freq_set_tuple] = support
        
        # Находим условные паттерны
        conditional_patterns = find_conditional_patterns(item, header_table)
        
        if conditional_patterns:
            # Считаем частоту товаров в условных паттернах
            conditional_counts = Counter()
            for pattern, count in conditional_patterns:
                for pattern_item in pattern:
                    conditional_counts[pattern_item] += count
            
            # Фильтруем по минимальной поддержке
            conditional_frequent = {item: count for item, count in conditional_counts.items()
                                  if count >= min_support_count}
            
            if conditional_frequent:
                # Строим условное дерево
                conditional_transactions = []
                for pattern, count in conditional_patterns:
                    # Берем только частые товары из паттерна
                    filtered_pattern = [p for p in pattern if p in conditional_frequent]
                    if filtered_pattern:
                        for _ in range(count):
                            conditional_transactions.append(filtered_pattern)
                
                if conditional_transactions:
                    # Подсчитываем частоту в условных транзакциях
                    cond_item_counts = Counter()
                    for trans in conditional_transactions:
                        for item in trans:
                            cond_item_counts[item] += 1
                    
                    # Строим условное FP-дерево
                    cond_root, cond_header, _ = build_fp_tree(
                        conditional_transactions, min_support_count, cond_item_counts)
                    
                    if cond_header:
                        # Рекурсивно добываем наборы
                        mine_fp_tree(cond_header, min_support_count, new_freq_set,
                                   frequent_itemsets, total_transactions, conditional_frequent)


def fp_growth(transactions, min_support=0.5):
    """
    Реализация алгоритма FP-Growth
    
    transactions: список транзакций (каждая транзакция - список товаров)
    min_support: минимальная поддержка (например, 0.5 = 50%)
    """
    total_transactions = len(transactions)
    min_support_count = int(min_support * total_transactions)
    
    # Подсчитываем частоту одиночных товаров
    item_counts = Counter()
    for transaction in transactions:
        for item in transaction:
            item_counts[item] += 1
    
    # Строим начальное FP-дерево
    root, header_table, frequent_items = build_fp_tree(
        transactions, min_support_count, item_counts)
    
    if root is None:
        return {1: {}}
    
    # Добываем частые наборы
    frequent_itemsets = {}
    prefix = []
    
    # Добавляем одиночные частые товары
    frequent_itemsets[1] = {}
    for item, count in sorted(frequent_items.items()):
        support = count / total_transactions
        frequent_itemsets[1][(item,)] = support
    
    # Добываем наборы большего размера
    mine_fp_tree(header_table, min_support_count, prefix, 
                frequent_itemsets, total_transactions, frequent_items)
    
    # Удаляем дубликаты и сортируем
    result = {}
    for k in sorted(frequent_itemsets.keys()):
        if frequent_itemsets[k]:
            # Удаляем наборы с поддержкой меньше min_support
            filtered = {itemset: support for itemset, support in frequent_itemsets[k].items()
                       if support >= min_support}
            
            if filtered:
                # Сортируем
                sorted_items = sorted(filtered.items(), 
                                     key=lambda x: (-x[1], x[0]))
                result[k] = dict(sorted_items)
                
                print(f"\nЧастые наборы из {k} товаров:")
                for itemset, support in result[k].items():
                    print(f"  {set(itemset)} - поддержка: {support:.2%}")
    
    return result


def calculate_support(itemset, transactions):
    """Считает поддержку для набора товаров"""
    count = 0
    itemset_set = set(itemset)
    for transaction in transactions:
        if itemset_set.issubset(set(transaction)):
            count += 1
    return count / len(transactions)


def generate_rules(frequent_items, transactions, min_confidence=0.7):
    """
    Генерирует правила ассоциации по частым наборам.
    """
    rules = []

    # Подготовим быстрый доступ к поддержкам
    support_lookup = {}
    for size_dict in frequent_items.values():
        support_lookup.update(size_dict)

    for k in frequent_items:
        if k < 2:
            continue

        for itemset, support_full in frequent_items[k].items():
            itemset_set = set(itemset)

            # Генерируем все возможные правила
            for i in range(1, k):
                for antecedent in combinations(itemset, i):
                    antecedent_set = set(antecedent)
                    consequent_set = itemset_set - antecedent_set

                    antecedent_tuple = tuple(sorted(antecedent_set))
                    support_antecedent = support_lookup.get(
                        antecedent_tuple,
                        calculate_support(antecedent_tuple, transactions)
                    )
                    confidence = support_full / support_antecedent if support_antecedent > 0 else 0

                    if confidence >= min_confidence:
                        consequent_tuple = tuple(sorted(consequent_set))
                        support_consequent = support_lookup.get(
                            consequent_tuple,
                            calculate_support(consequent_tuple, transactions)
                        )

                        lift = confidence / support_consequent if support_consequent > 0 else 0

                        rules.append({
                            'antecedent': antecedent_set,
                            'consequent': consequent_set,
                            'support': support_full,
                            'confidence': confidence,
                            'lift': lift
                        })

    return rules


def read_transactions_from_csv(file_path):
    """
    Читает транзакции из CSV файла.
    """
    transactions = []
    with open(file_path, 'r', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            cleaned_row = [item.strip() for item in row if item.strip()]
            if cleaned_row:
                transactions.append(cleaned_row)
    return transactions


# Тестовый пример с вашими данными
def test_with_hardcoded_data():
    # Ваши данные
    transactions = [
        ['хлеб', 'вода', 'молоко'],
        ['масло', 'вода', 'орехи'],
        ['шоколад', 'хлеб', 'масло', 'вода', 'орехи'],
        ['масло', 'молоко', 'орехи']
    ]

    print("Транзакции:")
    for i, t in enumerate(transactions, 1):
        print(f"{i}: {t}")

    print("\n" + "=" * 50)
    print("ЗАПУСК FP-GROWTH")
    print("=" * 50)

    # Используем min_support=0.5 (50%) как в вашем Apriori
    min_support = 0.5
    frequent_items = fp_growth(transactions, min_support)

    print("\n" + "=" * 50)
    print("ГЕНЕРАЦИЯ ПРАВИЛ")
    print("=" * 50)

    # Генерируем правила
    min_confidence = 0.3  # 60%
    rules = generate_rules(frequent_items, transactions, min_confidence)

    print(f"\nНайдено правил: {len(rules)}")
    for i, rule in enumerate(rules, 1):
        print(f"\nПравило {i}:")
        print(f"  {rule['antecedent']} → {rule['consequent']}")
        print(f"  Поддержка: {rule['support']:.2%}")
        print(f"  Достоверность: {rule['confidence']:.2%}")
        print(f"  Lift: {rule['lift']:.2f}")


# Пример использования
if __name__ == "__main__":
    # Тестируем на жестко заданных данных
    test_with_hardcoded_data()

Транзакции:
1: ['хлеб', 'вода', 'молоко']
2: ['масло', 'вода', 'орехи']
3: ['шоколад', 'хлеб', 'масло', 'вода', 'орехи']
4: ['масло', 'молоко', 'орехи']

ЗАПУСК FP-GROWTH

Частые наборы из 1 товаров:
  {'вода'} - поддержка: 75.00%
  {'масло'} - поддержка: 75.00%
  {'орехи'} - поддержка: 75.00%
  {'молоко'} - поддержка: 50.00%
  {'хлеб'} - поддержка: 50.00%

Частые наборы из 2 товаров:
  {'орехи', 'масло'} - поддержка: 75.00%
  {'вода', 'масло'} - поддержка: 50.00%
  {'вода', 'орехи'} - поддержка: 50.00%
  {'вода', 'хлеб'} - поддержка: 50.00%

Частые наборы из 3 товаров:
  {'вода', 'масло', 'орехи'} - поддержка: 50.00%

ГЕНЕРАЦИЯ ПРАВИЛ

Найдено правил: 14

Правило 1:
  {'масло'} → {'орехи'}
  Поддержка: 75.00%
  Достоверность: 100.00%
  Lift: 1.33

Правило 2:
  {'орехи'} → {'масло'}
  Поддержка: 75.00%
  Достоверность: 100.00%
  Lift: 1.33

Правило 3:
  {'вода'} → {'масло'}
  Поддержка: 50.00%
  Достоверность: 66.67%
  Lift: 0.89

Правило 4:
  {'масло'} → {'вода'}
  Поддержка: 50.00%
 